In [ ]:
!git clone https://github.com/jburroni/IntroPPLs26.git

%cd IntroPPLs26/notebooks/Jun-22

# Single-site Metropolis-Hastings

Three ways to sample the same model: likelihood weighting, independent MH, and single-site MH. The single-site cache is keyed by each sample's **execution index**.

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
from minippl import parse_one, PRIMITIVES, Symbol

## The evaluator

One recursive pass. At a `sample`, reuse the cached value for this index unless it is the chosen site `x0` or has none; otherwise draw. `observe` adds to the log-weight.

In [ ]:
class State:
    """An rng plus an index-keyed cache: the i-th sample reached gets key i."""
    def __init__(self, rng, x0=None, cache=None):
        self.rng = rng
        self.x0 = x0                 # sample index to resample (or None)
        self.cache = cache or {}     # {sample_index: value} from the previous trace
        self.i = 0                   # running sample index
        self.j = 0                   # running observe index
        self.X = {}                  # {sample_index: value}
        self.logP_s = {}             # {sample_index: log prob}
        self.logP_o = {}             # {observe_index: log prob}
        self.log_w = 0.0             # sum of observe log-probs

def evaluate(e, env, st):
    if isinstance(e, Symbol):
        return env[e]
    if not isinstance(e, list):
        return e
    op, *args = e
    if op == "let":
        binds, *body = args
        env = dict(env)
        for k in range(0, len(binds), 2):
            env[binds[k]] = evaluate(binds[k + 1], env, st)
        out = None
        for b in body:
            out = evaluate(b, env, st)
        return out
    if op == "if":
        test, then, els = args
        return evaluate(then, env, st) if evaluate(test, env, st) else evaluate(els, env, st)
    if op == "sample":
        d = evaluate(args[0], env, st)
        idx = st.i; st.i += 1                         # the index = execution order
        if idx != st.x0 and idx in st.cache:
            x = st.cache[idx]                         # reuse by index
        else:
            x = d.sample(st.rng)                      # draw: chosen site, or no cached value
        st.X[idx] = x; st.logP_s[idx] = d.log_prob(x)
        return x
    if op == "observe":
        d = evaluate(args[0], env, st)
        v = evaluate(args[1], env, st)
        lp = d.log_prob(v)
        st.logP_o[st.j] = lp; st.j += 1; st.log_w += lp
        return v
    fn = PRIMITIVES[op] if op in PRIMITIVES else evaluate(op, env, st)
    return fn(*[evaluate(a, env, st) for a in args])

def run(program, rng, x0=None, cache=None):
    st = State(rng, x0=x0, cache=cache)
    value = evaluate(parse_one(program), {}, st)
    return value, st

## The eight-bit model

Eight bits, each `1` when a `uniform(0,1)` falls below `0.5`. We observe their sum near `7`. The sum is integer, so the exact posterior is a short sum.

In [ ]:
bits = """
(let [b1 (if (< (sample (uniform-continuous 0 1)) 0.5) 1 0)
      b2 (if (< (sample (uniform-continuous 0 1)) 0.5) 1 0)
      b3 (if (< (sample (uniform-continuous 0 1)) 0.5) 1 0)
      b4 (if (< (sample (uniform-continuous 0 1)) 0.5) 1 0)
      b5 (if (< (sample (uniform-continuous 0 1)) 0.5) 1 0)
      b6 (if (< (sample (uniform-continuous 0 1)) 0.5) 1 0)
      b7 (if (< (sample (uniform-continuous 0 1)) 0.5) 1 0)
      b8 (if (< (sample (uniform-continuous 0 1)) 0.5) 1 0)
      total (+ b1 b2 b3 b4 b5 b6 b7 b8)]
  (observe (normal 7 1) total)
  total)
"""

from math import comb
w_exact = {k: comb(8, k) * math.exp(-0.5 * ((k - 7) / 1) ** 2) for k in range(9)}
Z = sum(w_exact.values())
exact_mean = sum(k * w_exact[k] for k in w_exact) / Z
print("exact   E[total] =", round(exact_mean, 3))

## Likelihood weighting

Run the program forward `L` times; weight each run by the likelihood. `ESS` is the effective number of runs.

In [ ]:
def likelihood_weighting(program, rng, L=200):
    xs, lw = zip(*((v, st.log_w) for v, st in (run(program, rng) for _ in range(L))))
    xs = np.array(xs, dtype=float)
    w = np.exp(np.array(lw) - max(lw)); w /= w.sum()
    return xs, w

xs_lw, w_lw = likelihood_weighting(bits, np.random.default_rng(0))
print("LW      E[total] =", round(float(np.sum(w_lw * xs_lw)), 3),
      "  ESS =", round(float(1.0 / np.sum(w_lw ** 2))), "/", len(w_lw))

## Independent MH

Propose a whole fresh run from the prior; accept with probability `min(1, W'/W)`.

In [15]:
def independent_mh(program, rng, S=200):
    x, st = run(program, rng)
    chain, acc = [float(x)], 0
    for _ in range(S):
        x2, st2 = run(program, rng)                         # fresh run from the prior
        if math.log(rng.random()) < st2.log_w - st.log_w:   # accept w.p. min(1, W'/W)
            x, st, acc = x2, st2, acc + 1
        chain.append(float(x))
    return np.array(chain), acc / S

chain_imh, acc_imh = independent_mh(bits, np.random.default_rng(1))
print("indMH   E[total] =", round(float(chain_imh.mean()), 3), "  accept =", round(acc_imh, 2))

indMH   E[total] = 5.861   accept = 0.28


## Single-site MH

Pick one sample index, resample it, reuse the rest from the cache. The acceptance ratio is the reused-cancelled trace ratio times `|X| / |X'|`.

In [19]:
def ssmh_accept(idx, X_prime, X, logP_s_prime, logP_o_prime, logP_s, logP_o):
    fwd = {idx} | (set(X_prime) - set(X))
    rev = {idx} | (set(X) - set(X_prime))

    num = sum(p for i, p in logP_s_prime.items() if i not in fwd) + sum(logP_o_prime.values())
    den = sum(p for i, p in logP_s.items() if i not in rev) + sum(logP_o.values())

    log_alpha = (math.log(len(X)) - math.log(len(X_prime))) + (num - den)
    return math.exp(log_alpha) if log_alpha < 0 else 1.0

def single_site_mh(program, rng, S=200):
    x, st = run(program, rng)
    chain, acc = [float(x)], 0

    for _ in range(S):
        idx = int(rng.integers(len(st.X)))
        x2, st2 = run(program, rng, x0=idx, cache=st.X)

        alpha = ssmh_accept(idx, st2.X, st.X, st2.logP_s, st2.logP_o, st.logP_s, st.logP_o)

        if rng.random() < alpha:
            x, st, acc = x2, st2, acc + 1

        chain.append(float(x))

    return np.array(chain), acc / S

chain_ss, acc_ss = single_site_mh(bits, np.random.default_rng(2))
print("SS-MH   E[total] =", round(float(chain_ss.mean()), 3), "  accept =", round(acc_ss, 2))

SS-MH   E[total] = 6.035   accept = 0.8


## Compare

All three estimate the same posterior over the sum. Plot it against the exact pmf.

In [ ]:
def pmf(values, weights=None):
    h = np.zeros(9)
    if weights is None:
        for v in values: h[int(round(v))] += 1
        h /= h.sum()
    else:
        for v, wv in zip(values, weights): h[int(round(v))] += wv
    return h

ks = np.arange(9)
plt.figure(figsize=(8, 3.2))
plt.bar(ks - 0.30, [w_exact[k] / Z for k in ks], width=0.2, label="exact")
plt.bar(ks - 0.10, pmf(xs_lw, w_lw),            width=0.2, label="LW")
plt.bar(ks + 0.10, pmf(chain_imh),              width=0.2, label="indep MH")
plt.bar(ks + 0.30, pmf(chain_ss),               width=0.2, label="SS-MH")
plt.xlabel("total"); plt.ylabel("P(total)"); plt.legend(); plt.show()

## Question

The cache keys each sample by its **execution index**. For the eight-bit model the samples always run in the same order, so that index is a stable name and all three methods agree.

Why is the index unsafe once control flow decides which samples run? Run the flip model below and compare to likelihood weighting.

## The flip model

A coin `z` decides whether `a` is drawn. When `z` is false, `a` is skipped, so `b` slides from index 2 to index 1.

In [ ]:
flip = "(let [z (sample (bernoulli 0.5)) a (if z (sample (normal 0 1)) 0) b (sample (normal 10 1))] (observe (normal (+ a b) 1) 10.0) z)"

xs_f, w_f = likelihood_weighting(flip, np.random.default_rng(0), L=40000)
print("LW      P(z=true) =", round(float(np.sum(w_f * xs_f)), 3))

z_chain, acc_f = single_site_mh(flip, np.random.default_rng(0))
print("SS-MH   P(z=true) =", round(float(z_chain.mean()), 3), "  accept =", round(acc_f, 2))
print("z changed", int(np.sum(z_chain[1:] != z_chain[:-1])), "times in", len(z_chain) - 1, "steps")